# 28 — Pretraining a bigram model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/028-pretraining.ipynb)

Pretraining is counting: read text, record how often each word follows each other word, and turn the counts into probabilities. This notebook trains that model on a handful of sentences, scores it with loss and perplexity, and shows why those two numbers only mean something next to a baseline.

## 1. The numbers

The corpus is one paragraph of very plain English. Each cell of the count table answers one question: how many times did the column's word follow the row's word?

In [ ]:
import numpy as np                                          # the array library, already on Colab

corpus = ("the cat sat on the mat . the dog ate the rat . the cat saw the dog . "   # first sentence
          "the rat saw the cat . the dog sat on the mat . the cat ate the rat .").split()   # a tiny corpus
held = "the cat sat on the mat . the dog saw the rat .".split()   # a passage kept aside for scoring
vocab = sorted(set(corpus + held))                          # every distinct word, alphabetically
index = {w: i for i, w in enumerate(vocab)}                 # word -> row number
V = len(vocab)                                              # the vocabulary size

counts = np.zeros((V, V))                                   # rows: first word, columns: next word
for a, b in zip(corpus, corpus[1:]):                        # every adjacent pair of words
    counts[index[a], index[b]] += 1                         # one more observation

print("vocabulary:", vocab)                                 # ten words, so V = 10
print("bigrams:", len(corpus) - 1)                          # one per adjacent pair
print(counts.astype(int))                                   # the whole table, as integers
print("the most common follower of 'the' is", vocab[int(counts[index['the']].argmax())])   # 'cat'

**Line by line**

- `import numpy as np` — load NumPy for the count table.
- `corpus = ....split()` — the training text, split into words on the spaces.
- `held = ....split()` — a separate passage; the model never counts on it.
- `vocab = sorted(set(...))` — each distinct word once, in alphabetical order.
- `index = {w: i ...}` — a dictionary mapping a word to its row number.
- `V = len(vocab)` — how many words the model must choose between.
- `counts = np.zeros((V, V))` — an empty table; entry (r, c) counts row word then column word.
- `for a, b in zip(corpus, corpus[1:])` — walk the text two words at a time.
- `counts[index[a], index[b]] += 1` — add one observation to that cell.
- `print(...)` — the vocabulary, the corpus size, and the table itself.

## 2. Step by step: counts to loss

A count becomes a probability when you divide by the row's total. The loss for one prediction is minus the logarithm of the probability the model gave the word that actually came next, so a confident mistake is expensive.

In [ ]:
lam = 0.2                                                   # share of the mass reserved for unseen words

def next_probs(c, word):                                    # one row of probabilities
    row = c[index[word]]                                    # counts of what followed this word
    return (1 - lam) * row / max(row.sum(), 1.0) + lam / V   # mix counts with a uniform floor

nll = 0.0                                                   # running total of the loss
pairs = list(zip(corpus, corpus[1:]))                       # the training bigrams

for step, (a, b) in enumerate(pairs[:6]):                   # show the first six, one at a time
    p = next_probs(counts, a)[index[b]]                     # probability given to the true next word
    nll += -np.log(p)                                       # surprise: minus log probability
    print(f"step {step}: {a:>3} -> {b:<3}  p={p:.3f}  loss={-np.log(p):.3f}")   # one line per step

for a, b in pairs[6:]:                                      # finish the remaining pairs
    nll += -np.log(next_probs(counts, a)[index[b]])         # no printing, same arithmetic

mean_nll = nll / len(pairs)                                 # loss per prediction
print("mean loss  =", round(mean_nll, 3))                   # in units of nats
print("perplexity =", round(float(np.exp(mean_nll)), 3))    # e raised to the loss

**Line by line**

- `lam = 0.2` — a fifth of the probability mass is shared out evenly.
- `def next_probs(c, word)` — the model's whole prediction, for one row.
- `row = c[index[word]]` — the counts of every word that followed this one.
- `return (1 - lam) * row / max(row.sum(), 1.0) + lam / V` — mostly counts, plus a flat floor.
- `nll = 0.0` — the accumulator for the total loss.
- `pairs = list(zip(...))` — the corpus as a list of (word, next word).
- `for step, (a, b) in enumerate(...)` — `enumerate` supplies a counter as well as the pair.
- `p = next_probs(...)[index[b]]` — look up the probability of the word that really came next.
- `nll += -np.log(p)` — add the surprise; a probability of 1 costs zero.
- `mean_nll = nll / len(pairs)` — average over predictions.
- `np.exp(mean_nll)` — perplexity: the loss converted back to a count of choices.

## 3. Plot: loss against corpus size

Train on longer and longer prefixes of the paragraph and score each version on the same held-out passage. With more text the counts stop being accidents, and the loss falls until the data runs out.

In [ ]:
import matplotlib.pyplot as plt                             # plotting library, already on Colab

def train(prefix):                                          # count bigrams in a prefix only
    c = np.zeros((V, V))                                    # a fresh table
    for a, b in zip(prefix, prefix[1:]):                    # adjacent pairs of the prefix
        c[index[a], index[b]] += 1                          # count them
    return c                                                # the table, unnormalised

def loss_on(c, passage):                                    # average loss on a passage
    total, n = 0.0, 0                                       # accumulator and counter
    for a, b in zip(passage, passage[1:]):                  # every pair in the passage
        total += -np.log(next_probs(c, a)[index[b]])        # same formula as section 2
        n += 1                                              # one more prediction
    return total / n                                        # loss per prediction

sizes = list(range(6, len(corpus) - 4))                     # prefix lengths from 6 words up
losses = [loss_on(train(corpus[:n]), held) for n in sizes]  # one loss per prefix length

plt.figure(figsize=(6, 3))                                  # a wide, short figure
plt.plot(sizes, losses, marker="o", color="tab:blue")       # loss against amount of training text
plt.xlabel("words of training text")                        # x-axis label
plt.ylabel("held-out loss (nats per word)")                 # y-axis label
plt.title("more text, lower loss - then the paragraph runs out")   # the takeaway
plt.show()                                                  # render it

print("first loss", round(losses[0], 3), "last loss", round(losses[-1], 3))   # the two ends

**Line by line**

- `import matplotlib.pyplot as plt` — load the plotting library.
- `def train(prefix)` — build a count table from a prefix only.
- `def loss_on(c, passage)` — score a passage with a given table.
- `total += -np.log(next_probs(c, a)[index[b]])` — the same surprise as before.
- `sizes = list(range(6, len(corpus) - 4))` — growing amounts of training text.
- `losses = [loss_on(...) for n in sizes]` — a list comprehension: one score per size.
- `plt.plot(sizes, losses, marker="o")` — the curve; markers show where data exists.
- `plt.title(...)` — the takeaway, not a description of the axes.

## 4. Interactive: slide the vocabulary size

The same counts read with a bigger word list. Every extra word takes a share of the reserved mass, so the words that do occur get slightly less, and the perplexity rises.

In [ ]:
from ipywidgets import interact                              # sliders that rerun a function

c_full = train(corpus)                                      # counts from the whole paragraph

def show(extra=0):                                          # extra words come from the slider
    Ve = V + extra                                          # vocabulary the model must choose from
    def probs(word):                                        # row of probabilities at this size
        row = c_full[index[word]]                           # the counts of this word's followers
        return (1 - lam) * row / row.sum() + lam / Ve       # same mix, a smaller floor
    p_the = probs("the")                                    # what tends to follow 'the'
    total, n = 0.0, 0                                       # score the held-out passage again
    for a, b in zip(held, held[1:]):                        # every pair in the held-out text
        total += -np.log(probs(a)[index[b]])                # surprise at this vocabulary size
        n += 1                                              # one more prediction
    plt.figure(figsize=(6, 3))                              # a fresh figure on every drag
    plt.bar(vocab, p_the, color="tab:blue")                 # one bar per real word
    plt.xticks(rotation=45)                                 # word labels would overlap otherwise
    plt.ylabel("probability of being next after 'the'")     # what the bars measure
    plt.title(f"V={Ve}: perplexity {np.exp(total / n):.2f}")   # live readout
    plt.show()                                              # draw the frame

interact(show, extra=(0, 70, 5))                            # attach the slider

**Line by line**

- `from ipywidgets import interact` — Colab ships the widget library.
- `c_full = train(corpus)` — count once; every frame reuses the table.
- `def show(extra=0)` — the slider adds unused words to the vocabulary.
- `Ve = V + extra` — the size the model must divide its probability over.
- `def probs(word)` — the smoothed row, computed for the current `Ve`.
- `return (1 - lam) * row / row.sum() + lam / Ve` — more words means a thinner floor.
- `for a, b in zip(held, held[1:])` — rescore the held-out passage at this size.
- `plt.bar(vocab, p_the)` — the whole prediction for the word 'the' as bars.
- `plt.title(f"V={Ve}: perplexity ...")` — the live readout.
- `interact(show, extra=(0, 70, 5))` — the slider, its range and step.

## 5. When it breaks: a uniform model never learns

A model that ignores the text and spreads its bet evenly over the vocabulary scores exactly the same on every passage. Its perplexity is the vocabulary size itself, which is why a perplexity number means nothing unless you say what the baseline was.

In [ ]:
print("corpus size | learned perplexity | uniform perplexity")   # a small table header
for n in sizes[::4]:                                            # a few corpus sizes, for brevity
    lp = np.exp(loss_on(train(corpus[:n]), held))               # the model trained on n words
    print(f"{n:11d} | {lp:18.2f} | {V:10.2f}")                  # the flat baseline on the right

plt.figure(figsize=(6, 3))                                      # a wide, short figure
plt.plot(sizes, np.exp(losses), marker="o", color="tab:blue", label="bigram model")   # learned
plt.plot(sizes, np.full(len(sizes), V), color="tab:red", label="uniform model")       # flat baseline
plt.xlabel("words of training text")                            # x-axis label
plt.ylabel("perplexity (choices of next word)")                 # y-axis label
plt.legend()                                                    # name the two lines
plt.title("the uniform baseline is stuck at V = 10, whatever it reads")   # the takeaway
plt.show()                                                      # render it

print("every uniform prediction has probability", round(1 / V, 3), "so the loss is", round(np.log(V), 3))   # ln V

**Line by line**

- `print("...")` — a header row for the table.
- `for n in sizes[::4]` — every fourth size, so the table stays short.
- `lp = np.exp(loss_on(...))` — perplexity of the model trained on `n` words.
- `print(f"...")` — `:11d` and `:18.2f` line the columns up.
- `plt.plot(sizes, np.exp(losses), ...)` — the learned curve from section 3.
- `np.full(len(sizes), V)` — a constant array, which draws a flat line.
- `plt.legend()` — label which line is which.
- `np.log(V)` — a uniform model's loss, exactly the log of the vocabulary size.

## Your turn

Set `lam = 0.0` in section 2 and rerun section 3: held-out loss becomes enormous because any unseen pair now gets probability zero. Then predict what section 5's learned curve does as `lam` approaches 1 — and check against the uniform baseline once it does.